# Connect to Neo4j � deluxe-analyze

**Current Neo4j IP:** `34.28.39.13`  
**Bolt port:** `7687`  
**HTTP port:** `7474`

## Firewall / Access requirements

The Neo4j VM firewall (`allow-neo4j-bolt`) whitelists the range `181.78.0.0/24`.  
Your client IP must fall within that range, **or** you must connect through a Cloud IAP tunnel:

```bash
# Open IAP tunnel � run in a terminal, keep it running
gcloud compute start-iap-tunnel neo4j-vm 7687 \n
  --local-host-port=localhost:7687 \n
  --project=engaged-stage-463123-e0 \n
  --zone=us-central1-a
```

Then set `NEO4J_IP = "localhost"` in the cell below.

## Password

Retrieve the password from GCP Secret Manager � do **not** hardcode it:

```bash
gcloud secrets versions access latest \n
  --secret=neo4j-password \n
  --project=engaged-stage-463123-e0
```

Or set `NEO4J_PASSWORD` as an environment variable before launching the notebook.


In [1]:
import os
from neo4j import GraphDatabase

# Direct connection � works if your IP is in 181.78.0.0/24
# Use 'localhost' if you have an active Cloud IAP tunnel on port 7687
NEO4J_IP = os.environ.get("NEO4J_IP", "34.28.39.13")

# Set NEO4J_PASSWORD env var before running, or retrieve from Secret Manager:
# gcloud secrets versions access latest --secret=neo4j-password --project=engaged-stage-463123-e0
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "")
if not NEO4J_PASSWORD:
    raise ValueError("Set NEO4J_PASSWORD env var or retrieve from Secret Manager")

driver = GraphDatabase.driver(
    f"bolt://{NEO4J_IP}:7687",
    auth=("neo4j", NEO4J_PASSWORD),
)
driver.verify_connectivity()
print("Connected to Neo4j")


Connected to Neo4j


In [6]:
with driver.session(database="neo4j") as session:
    session.run("CALL gds.graph.drop('social-graph', false)")
print("OK, listo para re-proyectar")

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL gds.graph.drop('social-graph', false)"


OK, listo para re-proyectar


In [7]:
# Verify seed data is loaded
with driver.session(database="neo4j") as session:
    result = session.run(
        "MATCH (n) RETURN labels(n)[0] AS label, count(n) AS cnt ORDER BY cnt DESC"
    )
    print("Node counts by label:")
    for row in result:
        print(f"  {row['label']}: {row['cnt']}")


Node counts by label:
  Usuario: 361
  Mesa: 59
  Evento: 35


In [11]:
with driver.session(database="neo4j") as session:
    # Basic connectivity check
    result = session.run("MATCH (n) RETURN count(n) AS total")
    print(f"Total nodes: {result.single()['total']}")

    # Degree centrality via GDS
    result = session.run("""
        CALL gds.degree.stream('social-graph')
        YIELD nodeId, score
        RETURN gds.util.asNode(nodeId).id AS user_id, score AS degree
        ORDER BY degree DESC
        LIMIT 10
    """)
    for row in result:
        print(f"  {row['user_id']}: degree={row['degree']}")

Total nodes: 455
  csv:356: degree=332.0
  csv:328: degree=331.0
  csv:46: degree=330.0
  csv:10: degree=328.0
  csv:164: degree=327.0
  csv:291: degree=325.0
  csv:324: degree=325.0
  csv:119: degree=324.0
  csv:87: degree=324.0
  csv:1: degree=323.0


In [9]:
# Crear proyeccion del grafo social
with driver.session(database="neo4j") as session:
    # Proyeccion en memoria
    session.run("""
        CALL gds.graph.project(
            'social-graph',
            'Usuario',
            {
                CONOCE_A: {
                    orientation: 'UNDIRECTED',
                    properties: ['tie_strength']
                }
            }
        )
    """)

    # Louvain community detection
    result = session.run("""
        CALL gds.louvain.stream('social-graph')
        YIELD nodeId, communityId
        RETURN communityId, count(*) AS members
        ORDER BY members DESC
        LIMIT 10
    """)

    print("Top 10 comunidades (Louvain):")
    for row in result:
        print(f"  comunidad {row['communityId']}: {row['members']} miembros")

Top 10 comunidades (Louvain):
  comunidad 251: 103 miembros
  comunidad 23: 97 miembros
  comunidad 27: 67 miembros
  comunidad 220: 51 miembros
  comunidad 242: 22 miembros
  comunidad 30: 1 miembros
  comunidad 6: 1 miembros
  comunidad 71: 1 miembros
  comunidad 108: 1 miembros
  comunidad 31: 1 miembros


In [10]:
# Degree centrality — usuarios mas conectados
with driver.session(database="neo4j") as session:
    result = session.run("""
        CALL gds.degree.stream('social-graph')
        YIELD nodeId, score
        MATCH (u:Usuario) WHERE id(u) = nodeId
        RETURN u.id AS user_id, score AS degree
        ORDER BY degree DESC
        LIMIT 10
    """)

    print("Top 10 usuarios por grado (CONOCE_A):")
    for row in result:
        print(f"  {row['user_id']}: {int(row['degree'])} conexiones")

Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=33, offset=108>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 108, 'line': 4, 'column': 33}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL gds.degree.stream('social-graph')\n        YIELD nodeId, score\n        MATCH (u:Usuario) WHERE id(u) = nodeId\n        RETURN u.id AS user_id, score AS degree\n        ORDER BY degree DESC\n        LIMIT 10\n    "


Top 10 usuarios por grado (CONOCE_A):
  csv:356: 332 conexiones
  csv:328: 331 conexiones
  csv:46: 330 conexiones
  csv:10: 328 conexiones
  csv:164: 327 conexiones
  csv:291: 325 conexiones
  csv:324: 325 conexiones
  csv:119: 324 conexiones
  csv:87: 324 conexiones
  csv:1: 323 conexiones


## Firewall whitelist � `181.78.0.0/24`

The GCP firewall rule `allow-neo4j-bolt` permits Bolt (port 7687) only from `181.78.0.0/24`.  
If your public IP is outside that range you have two options:

**Option A � Add your IP to the whitelist** (requires `terraform apply`):
1. Edit `infra/terraform.tfvars` and add your IP/CIDR to `allowed_neo4j_ips`.
2. Run `terraform apply` from the `infra/` directory.

**Option B � Cloud IAP tunnel** (no firewall change needed, recommended):
```bash
gcloud compute start-iap-tunnel neo4j-vm 7687 \n
  --local-host-port=localhost:7687 \n
  --project=engaged-stage-463123-e0 \n
  --zone=us-central1-a
```
Then set `NEO4J_IP = "localhost"` in the connection cell.

**Option C � Colab** (rotating IPs):
Colab IPs change between sessions. Use IAP tunnel via a local proxy, or add the Colab IP range to the whitelist each session.
